# LLM-in-Sandbox on 2x T4

Reproduction driver for [arXiv:2601.16206](https://arxiv.org/abs/2601.16206).

Every long-running step is launched **detached** (`nohup ... &`) and polled from a
separate cell. That is deliberate: a notebook cell that blocks on a server or a
multi-hour sweep cannot be interrupted when the notebook is being driven over an
MCP bridge, and a wedged cell blocks the kernel for everything else.

Runtime: **GPU, 2x T4**.

## 1. What are we actually on?

In [ ]:
import subprocess, torch, os
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,compute_cap',
                      '--format=csv'], capture_output=True, text=True).stdout)
print('cpus', os.cpu_count(), '| torch', torch.__version__)
cap = torch.cuda.get_device_capability(0)
print('compute capability', cap)
if cap < (8, 0):
    print('=> sm<80: fp16 only, no FlashAttention-2/FP8/Marlin. Force TRITON_ATTN.')

## 2. Install

`transformers` is pinned to 4.x: Colab ships 5.0, which removed
`all_special_tokens_extended`, and vLLM 0.11 still calls it. vLLM's own pin
(`>=4.55.2`) is satisfied by 5.x, so pip will not downgrade it for you.

In [ ]:
%pip install -q vllm==0.11.0 'transformers==4.56.2' 'tokenizers<0.23'
print('restart the runtime if vLLM was already imported in this session')

In [ ]:
!git clone -q https://github.com/Maverick-Ansh/llm-in-sandbox.git /content/llm-in-sandbox || \
  (cd /content/llm-in-sandbox && git pull -q)
%pip install -q -e '/content/llm-in-sandbox[dev]'
!cd /content/llm-in-sandbox && python -m pytest tests/ -q --no-header -rs 2>&1 | tail -15

## 3. Serve one replica per GPU

One replica per card, not tensor-parallel: the two T4s are `PHB` (PCIe host
bridge, no NVLink), so TP=2 spends its time moving activations, while agent
episodes parallelise across replicas for free.

In [ ]:
!cd /content/llm-in-sandbox && python scripts/serve.py \
  --model Qwen/Qwen3-4B-Instruct-2507 --served-name qwen3-4b \
  --gpus 0 1 --max-model-len 24576 --wait 900

## 4. Sweep

Resumable: re-running the same command skips completed episodes, so a dropped
runtime costs one episode rather than the whole run.

In [ ]:
import subprocess, textwrap, os
os.makedirs('/content/logs', exist_ok=True)
open('/content/sweep.sh','w').write(textwrap.dedent('''
cd /content/llm-in-sandbox
export PYTHONPATH=/content/llm-in-sandbox/src HF_HUB_DISABLE_PROGRESS_BARS=1
python scripts/run_sweep.py \\
  --benchmark mmlu_pro --n 25 --model qwen3-4b \\
  --base-url http://127.0.0.1:8000/v1 http://127.0.0.1:8001/v1 \\
  --modes direct sandbox --max-turns 20 --workers 8 \\
  --out /content/runs/qwen3-4b 2>&1
echo ___SWEEP_DONE___
'''))
subprocess.Popen('nohup bash /content/sweep.sh > /content/logs/sweep.log 2>&1 &', shell=True)
print('sweep launched -> /content/logs/sweep.log')

In [ ]:
# Re-run this cell to watch progress.
import subprocess, json, pathlib, collections
print(subprocess.run('tail -c 900 /content/logs/sweep.log', shell=True,
                     capture_output=True, text=True).stdout)
p = pathlib.Path('/content/runs/qwen3-4b/results.jsonl')
if p.exists():
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    for mode in ('direct','sandbox'):
        sub = [r for r in rows if r['mode'] == mode]
        if sub:
            print(f"{mode:8s} n={len(sub):3d} "
                  f"acc={sum(r['correct'] for r in sub)/len(sub):.3f} "
                  f"tok={sum(r['generated_tokens'] for r in sub)/len(sub):7.0f}")
    print(collections.Counter(r['stop_reason'] for r in rows).most_common())

## 5. Report

Deltas come with paired bootstrap CIs and an exact McNemar p. At tens of items
per domain a bare delta is a couple of questions changing hands.

In [ ]:
!cd /content/llm-in-sandbox && python scripts/report.py \
  /content/runs/qwen3-4b/results.jsonl --out /content/runs/qwen3-4b/report.md

## 6. Enforced capability ablation

Colab denies `unshare(2)` even to uid 0, so the network ablation is enforced
with a seccomp-BPF filter instead. Check which backends this host supports
before trusting an ablation result.

In [ ]:
from sandbox_lab.sandbox import seccomp_available
import subprocess
print('unshare:', subprocess.run('unshare --fork --pid --mount true', shell=True,
                                 capture_output=True, text=True).returncode == 0)
print('seccomp:', seccomp_available())

In [ ]:
# Sandbox with external resource access removed for real (not just discouraged).
!cd /content/llm-in-sandbox && python scripts/run_sweep.py \
  --benchmark mmlu_pro --n 25 --model qwen3-4b \
  --base-url http://127.0.0.1:8000/v1 http://127.0.0.1:8001/v1 \
  --modes sandbox --caps nonet --backend seccomp --workers 8 \
  --out /content/runs/qwen3-4b